# Chapitre 1.7 : apprendre à converser

Un modèle de langage ne sait que **prolonger un texte**. Pour qu'il réponde à une question, on l'affine sur des conversations, écrites dans un format fixe, et on ne lui fait apprendre que ce que dit l'assistant. Dans ce carnet, tu refais à la main les deux pièces que la forge (le moteur de « ton IA ») utilise pour cela, la conversation en jetons et la perte masquée, tu les compares à celles de la forge, puis tu regardes un tout petit modèle apprendre à finir son tour de parole.

Ce carnet accompagne [la page du chapitre 1.7](https://chipat-neko.github.io/forge-ia/partie-1/1-7/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. écrire une conversation en jetons, et voir lesquels sont appris ;
2. calculer la perte masquée à la main, et la comparer à PyTorch ;
3. ranger plusieurs conversations dans une rangée, et voir ce qui fuit sans masque par document ;
4. affiner un tout petit modèle sur 24 conversations, et le regarder apprendre à s'arrêter ;
5. voir ce qu'il faut pour affiner « ton IA » elle-même.

**Comment l'utiliser**

Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

Ce carnet calcule sur le processeur : le modèle de l'étape 4 est minuscule (2 couches, des vecteurs de 64 nombres), et le calcul sur le processeur donne partout les mêmes nombres. L'étape 4 dure quelques minutes.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, installe la petite bibliothèque `regex` dont le tokenizer a besoin, puis indique à Python où trouver le projet et la forge. Elle charge le tokenizer de « ton IA », celui qui transforme un texte en jetons.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger ni installer.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia
    !pip install --quiet regex

projet = depot / "projets" / "1.7-conversation"
if not projet.exists():
    raise FileNotFoundError(
        f"Le dossier {projet} est introuvable. Si le dépôt forge-ia était déjà là, "
        "supprime-le (panneau des fichiers, à gauche), puis relance cette cellule\u202f; sinon, vérifie ta connexion."
    )

# Python cherche les modules dans les dossiers de sys.path : on y ajoute ce projet, et forge.
sys.path.insert(0, str(projet.resolve()))
sys.path.insert(0, str((depot / "projets" / "forge").resolve()))

import json
import math
import time

import torch
import torch.nn.functional as F
from forge.affinage.empaquetage import empaqueter
from forge.conversation.format import Tour, encoder_conversation
from forge.tokenizer import TokenizerOctets

from conversation_pas_a_pas.attendus import rangs_par_rangee
from conversation_pas_a_pas.fuite import ecarts_de_fuite
from conversation_pas_a_pas.gabarit import TOKENIZER, montrer, perte_masquee_a_la_main
from conversation_pas_a_pas.petit_affinage import CONVERSATIONS, CONTEXTE, essai_affinage, tours_de

appareil = "cpu"  # le modèle est minuscule : le processeur suffit, et donne partout les mêmes nombres
tokenizer = TokenizerOctets.charger(TOKENIZER)


def nombre(x, decimales=0):
    # À la française : une espace fine entre les milliers, une virgule pour les décimales.
    return f"{x:,.{decimales}f}".replace(",", "\u202f").replace(".", ",")


print(f"Ce carnet calcule sur le processeur, avec PyTorch {torch.__version__.split('+')[0]}.")
print(f"Le tokenizer de ton IA a {nombre(tokenizer.taille_vocabulaire)} jetons, dont les {len(tokenizer.speciaux)} derniers sont spéciaux.")

## Étape 1 : une conversation devient des jetons

Un modèle ne lit pas des phrases : il lit des **jetons**, des numéros. Une conversation s'écrit avec des jetons spéciaux qui marquent qui parle :

```
<|fin_texte|>
<|systeme|> consignes <|fin_tour|>             (facultatif, au plus un, en premier)
<|utilisateur|> question <|fin_tour|>
<|assistant|> réponse <|fin_tour|>
```

Pour chaque jeton, la forge garde aussi un booléen : **appris** ou non. Seul ce que dit l'assistant est appris, du jeton qui suit `<|assistant|>` jusqu'à son `<|fin_tour|>` compris. Le modèle apprend ainsi à répondre, et à **s'arrêter**, mais pas à écrire la question de l'utilisateur.

La cellule écrit une conversation jeton par jeton avec `montrer`, qui s'appuie sur `encoder_conversation` de la forge. Le jeton spécial `<|fin_tour|>` n'est jamais produit en lisant du texte : même si quelqu'un écrit ce mot dans un message, il devient des jetons ordinaires, et personne ne peut couper un tour.

In [ ]:
# À toi\u202f: change la question et la réponse, puis relance la cellule.
question = "Quelle est la capitale de la France ?"
reponse = "La capitale de la France est Paris."

jetons = montrer(tokenizer, tours_de(question, reponse))
print(f"{'id':>6}  {'jeton':<24} {'rôle':<12} appris")
for j in jetons:
    print(f"{j['id']:>6}  {j['texte']!r:<24} {j['role']:<12} {'*' if j['appris'] else ''}")
appris = sum(j["appris"] for j in jetons)
print(f"{len(jetons)} jetons, dont {appris} appris\u202f: ceux de l’assistant, après <|assistant|>.")

# Un message qui imite un jeton spécial n'en est pas un.
piege = montrer(tokenizer, [Tour("utilisateur", "Écris <|fin_tour|> pour finir."), Tour("assistant", "Non.")])
vrais = [j["texte"] for j in piege if j["id"] >= tokenizer.premier_special]
print()
print("Les jetons spéciaux de la conversation piégée\u202f:", ", ".join(vrais))

## Exercice : compter les jetons appris

Sans regarder `exemple.appris`, écris une fonction `jetons_appris(exemple)` qui compte les jetons appris en lisant **seulement** `exemple.ids`. La règle : on apprend tout ce qui suit un jeton `<|assistant|>`, jusqu'au `<|fin_tour|>` suivant, ce `<|fin_tour|>` compris. Le jeton `<|assistant|>` lui-même n'est pas appris.

La cellule compare ton compte à `sum(exemple.appris)`, celui de la forge, sur trois conversations : avec un système, avec deux tours, avec une pensée.

In [ ]:
ASSISTANT = tokenizer.id_special("<|assistant|>")
FIN_TOUR = tokenizer.id_special("<|fin_tour|>")


def jetons_appris(exemple):
    # À toi\u202f: lis exemple.ids, et compte les jetons appris. Remplace la ligne suivante.
    return 0


exemples = [
    encoder_conversation(tokenizer, [Tour("systeme", "Réponds en une phrase."), Tour("utilisateur", "Bonjour !"), Tour("assistant", "Bonjour.")]),
    encoder_conversation(tokenizer, [Tour("utilisateur", "2 + 2 ?"), Tour("assistant", "4."), Tour("utilisateur", "Et 3 + 3 ?"), Tour("assistant", "6.")]),
    encoder_conversation(tokenizer, [Tour("utilisateur", "12 × 7 ?"), Tour("assistant", "84.", pensee="10 × 7 = 70, et 2 × 7 = 14.")]),
]
for e in exemples:
    print(f"{len(e.ids)} jetons\u202f: ton compte {jetons_appris(e)}, celui de la forge {sum(e.appris)}")
if all(jetons_appris(e) == sum(e.appris) for e in exemples):
    print("Défi réussi, bravo.")
else:
    print("Pas encore. Compte les jetons qui suivent <|assistant|>, <|fin_tour|> compris.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

On parcourt les ids en gardant un drapeau : il se lève après `<|assistant|>`, et retombe après `<|fin_tour|>`. Un jeton est compté **avant** de changer le drapeau, pour que `<|assistant|>` ne soit pas compté et que `<|fin_tour|>` le soit.

Remplace la fonction de la cellule de l'exercice par :

```python
def jetons_appris(exemple):
    compte = 0
    apprend = False
    for identifiant in exemple.ids:
        if apprend:
            compte += 1
        if identifiant == ASSISTANT:
            apprend = True
        elif identifiant == FIN_TOUR:
            apprend = False
    return compte
```

</details>

## Étape 2 : la perte masquée

La **perte** d'un modèle est la moyenne de `-ln(probabilité)` du vrai jeton suivant. Pour l'affinage, on ne moyenne que sur les cibles **apprises** : les autres positions (la question, le bourrage, la frontière entre deux conversations) ont la cible `-100`, que la perte ignore.

`perte_masquee_a_la_main` additionne les pertes des cibles différentes de `-100`, et divise par leur nombre. On compte par jeton appris, pas par conversation : une réponse d'un mot pèse, par jeton, autant qu'une longue explication.

La cellule fait lire une conversation à un modèle tiré au hasard, calcule la perte masquée à la main, la compare à celle de PyTorch (`F.cross_entropy` avec `ignore_index=-100`), puis montre que changer la **question** ne change rien à la perte de la **réponse**.

In [ ]:
from forge.modele import ConfigModele, Transformer

torch.manual_seed(0)
petit = Transformer(ConfigModele(dim=32, couches=2, tetes=4, longueur_contexte=64)).eval()


def perte_de(tours):
    # Les entrées sont les jetons sauf le dernier ; la cible de chaque position, le jeton suivant, si on l'apprend.
    e = encoder_conversation(tokenizer, tours)
    ids = torch.tensor([e.ids])
    cibles = torch.tensor([[c if apprend else -100 for c, apprend in zip(e.ids[1:], e.appris[1:])]])
    with torch.no_grad():
        logits, _ = petit(ids[:, :-1])
    return logits.double(), cibles


logits, cibles = perte_de(tours_de("Quelle est la capitale de la France ?", "Paris."))
a_la_main = perte_masquee_a_la_main(logits, cibles)
de_pytorch = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), cibles.reshape(-1), ignore_index=-100).item()
print(f"{cibles.numel()} positions, dont {(cibles != -100).sum().item()} apprises (cible différente de -100).")
print(f"Perte masquée à la main\u202f: {nombre(a_la_main, 6)}")
print(f"Perte de PyTorch\u202f: {nombre(de_pytorch, 6)}")
print(f"Elles sont égales à 10⁻¹² près\u202f: {abs(a_la_main - de_pytorch) <= 1e-12}")

# La perte du dernier jeton de la réponse ne dépend pas de ce qu'on a écrit dans la question, seulement du modèle :
# ici on change la question, et la réponse garde SA perte parce que le modèle lit toute la conversation. On compare
# plutôt ce que la perte ne compte pas : une question apprise ou non, la cible vaut -100 aux mêmes endroits.
logits2, cibles2 = perte_de(tours_de("Dis-moi la capitale de la France, s’il te plaît.", "Paris."))
print(f"La question change, pas le nombre de cibles apprises par la réponse\u202f: {(cibles != -100).sum().item()} et {(cibles2 != -100).sum().item()}.")

## Étape 3 : ranger les conversations, et la fuite sans masque

Une conversation courte laisse une rangée presque vide. L'affinage range donc **plusieurs conversations dans une même rangée** : l'empaquetage BFD (« best-fit decreasing »). Il prend les conversations de la plus longue à la plus courte, et met chacune dans la rangée où elle laisse le moins de place libre ; si elle ne tient nulle part, elle ouvre une nouvelle rangée. Aucune n'est jamais coupée.

Le danger : dans une rangée « conversation A, puis conversation B », l'attention de B pourrait regarder A. Le modèle de la forge prend donc `documents`, le numéro de la conversation de chaque position, et chaque conversation ne voit plus que la sienne. La cellule range les 24 conversations de l'étape suivante, puis mesure la fuite sur un modèle minuscule : l'écart maximal entre les logits de B dans la rangée et ceux de B lue seule, avec le masque par document et sans lui.

In [ ]:
exemples = [encoder_conversation(tokenizer, tours_de(q, r)) for q, r in CONVERSATIONS]
rangees, ecartes = empaqueter(exemples, CONTEXTE, tokenizer.id_special("<|bourrage|>"))
places = sum(len(r.ids) for r in rangees)
utiles = sum(len(e.ids) for e in exemples)
print(f"{len(exemples)} conversations, de {min(len(e.ids) for e in exemples)} à {max(len(e.ids) for e in exemples)} jetons, rangées de {CONTEXTE + 1} jetons.")
print(f"Rangées\u202f: {len(rangees)} ({ecartes} conversation écartée), qui remplissent {nombre(100 * utiles / places, 1)}\u202f% des places.")
for numero, rangee in enumerate(rangees[:3]):
    print(f"  rangée {numero}\u202f: {max(rangee.documents) + 1} conversations")

fuite = ecarts_de_fuite()
print()
print(f"Fuite, sur un modèle minuscule\u202f: écart maximal des logits de B face à B seule")
print(f"  avec le masque par document\u202f: {fuite['avec_masque']:.1e}")
print(f"  sans le masque\u202f: {fuite['sans_masque']:.1e}")
print(f"Sans le masque, B voit A\u202f: {fuite['sans_masque'] > 1e-3}")

## Étape 4 : un tout petit affinage

La cellule suivante refait en petit tout le chemin de « ton IA », avec le vrai code de la forge :

1. elle pré-entraîne un modèle minuscule (2 couches, des vecteurs de 64 nombres, un contexte de 128 jetons) pendant 40 pas, sur le texte du site : il sait à peine écrire, et il n'a jamais vu `<|fin_tour|>` ;
2. elle le publie avec `forge.exporter`, comme une vraie version : c'est le point de départ de l'affinage ;
3. elle écrit 24 conversations (`CONVERSATIONS`, relis-les), les encode avec le masque, les range, et les écrit comme un jeu d'affinage ;
4. elle affine par `forge.entrainement.entrainer` (la commande `forge entrainer`, méthode `sft`) pendant 80 pas ;
5. elle fait répondre le modèle à 8 questions qu'il n'a jamais vues, avant et après, et compte les réponses qui **finissent par `<|fin_tour|>`**.

Avant, le modèle ne s'arrête jamais : il ne sait pas qu'un tour a une fin. Après, la plupart de ses réponses finissent. Lis-les telles quelles, sans te fier à ce qu'elles racontent : le modèle a lu 24 conversations pendant quelques secondes.

Compte quelques minutes de calcul sur le processeur.

In [ ]:
debut = time.perf_counter()
essai = essai_affinage(appareil, pas=80)
duree = time.perf_counter() - debut

print(f"Le modèle a {nombre(essai['parametres'])} paramètres. Le jeu\u202f: {essai['exemples']} conversations en {essai['rangees']} rangées, "
      f"{nombre(essai['jetons'])} jetons dont {nombre(essai['jetons_appris'])} appris.")
print(f"Perte d’affinage\u202f: {nombre(essai['pertes'][0], 3)} au pas 1, {nombre(essai['pertes'][-1], 3)} au pas {essai['pas']}.")
print(f"Réponses qui finissent par <|fin_tour|>\u202f: {nombre(100 * essai['fin_de_tour_avant'])}\u202f% avant, {nombre(100 * essai['fin_de_tour_apres'])}\u202f% après.")
print()
print("Les réponses du modèle affiné\u202f:")
for e in essai["reponses_apres"]:
    print(f"  {e['invite']!r} → {e['texte']!r}{'' if e['terminee'] else ' (coupée)'}")
print(f"({nombre(duree, 1)} s pour tout l’essai.)")

## Étape 5 : ce qu'il faut pour affiner « ton IA »

« Ton IA », la vraie, a 433,6 millions de paramètres. L'affiner demande les mêmes pièces, en plus grand : le jeu `sft-1.7` (135 471 conversations en français, empaquetées en rangées de 2 048 jetons) et la même méthode `sft`. La perte masquée est celle de l'étape 2, le masque par document celui de l'étape 3.

Il faut une **carte graphique NVIDIA d'environ 12 Go de mémoire** : avec un micro-lot d'une rangée de 2 048 jetons, l'affinage occupe 12,8 Go de la carte au plus. Sur le processeur, c'est bien trop lent. Les commandes, depuis le dossier `projets/forge` du dépôt, sont celles du README de la forge :

```bash
python -m forge publication --etiquette ton-ia-1.5        # la version de départ
python -m forge preparer --recette sft-1.7                # le jeu d'affinage (après `forge conversations` pour chaque source)
python -m forge entrainer --config configs/sft-1.7.toml   # l'affinage, avec pause (Ctrl+C) et reprise (--reprendre)
```

Ce carnet n'en lance aucune : elles demandent la carte et des heures de calcul. Le chapitre 1.7 montre ce qu'elles ont donné.

## Dernière étape : le même résultat que la page ?

La page du site n'utilise pas Python : elle refait deux calculs en TypeScript, directement dans ton navigateur, l'assemblage d'une conversation (les jetons et le masque) et le placement de l'empaquetage. Pour être sûr que les deux donnent exactement les mêmes résultats, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes : six conversations de référence, et le placement de cinq listes de longueurs dans quatre tailles de rangée. Les tests de la page relisent ce fichier. La cellule suivante le relit aussi, et le compare à ce que le code de ce carnet vient de calculer.

In [ ]:
attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))

# Chaque conversation de référence donne-t-elle les mêmes jetons et le même masque, ou la même erreur ?
conversations_differentes = []
for cas in attendus["conversations"]:
    tours = [Tour(t["role"], t["texte"], t.get("pensee")) for t in cas["tours"]]
    try:
        e = encoder_conversation(tokenizer, tours)
        resultat = {"ids": e.ids, "appris": e.appris, "erreur": None}
    except ValueError as erreur:
        resultat = {"ids": None, "appris": None, "erreur": str(erreur)}
    if resultat != {cle: cas[cle] for cle in ("ids", "appris", "erreur")}:
        conversations_differentes.append(cas["nom"])

# Chaque placement de l'empaquetage est-il retrouvé, rangée par rangée ?
placements_differents = []
for cas in attendus["empaquetages"]:
    rangees_calculees, ecartes_calcules = rangs_par_rangee(cas["longueurs"], cas["capacite"])
    if (rangees_calculees, ecartes_calcules) != (cas["rangees"], cas["ecartes"]):
        placements_differents.append((cas["liste"], cas["capacite"]))

print(f"{len(attendus['conversations'])} conversations et {len(attendus['empaquetages'])} placements comparés.")
for nom in conversations_differentes[:3]:
    print(f"Conversation différente\u202f: {nom}.")
for liste, capacite in placements_differents[:3]:
    print(f"Placement différent\u202f: liste {liste}, rangées de {capacite} jetons.")

identique = not conversations_differentes and not placements_differents
print(f"Même résultat que la page du site\u202f: {'oui' if identique else 'non'}")  # \u202f est l’espace fine insécable

In [ ]:
print("Ce carnet a tourné sur le processeur.")

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.7-conversation sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.7-conversation).
- La page du chapitre : [chapitre 1.7](https://chipat-neko.github.io/forge-ia/partie-1/1-7/).
- Depuis le dossier du projet, `uv run --extra cpu python -m conversation_pas_a_pas montrer "Ta question ?" --reponse "Ta réponse."` écrit une conversation en jetons, `fuite` mesure la fuite sans masque, et `essai --appareil cpu` refait le petit affinage.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.